# abalone9-18: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('08_abalone9-18_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,18.214,14.000,40.000,8.889,100.000,54.444,63.668,90.152,16.107
smote,74.971,43.435,33.775,62.222,92.511,77.367,55.860,92.129,59.203
bsmote,72.454,46.350,39.846,56.389,94.695,75.542,55.807,89.120,59.402
ros,82.461,50.552,39.167,74.028,92.296,83.162,60.463,93.313,66.506
adasyn,78.201,45.738,35.225,67.083,92.006,79.545,54.951,92.354,61.969
casb,80.324,52.967,43.716,69.167,94.182,81.674,62.226,93.561,66.646
casb_relaxed,78.996,50.839,42.056,68.056,93.963,81.009,60.894,92.913,64.918
casb_edited,80.278,48.744,38.369,70.556,92.436,81.496,57.039,92.558,64.511
casb_ros,77.618,51.245,43.314,65.417,94.474,79.945,65.122,93.830,64.431


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,32.594,27.268,60.000,22.500,100.000,61.250,45.697,79.806,29.931
smote,72.032,40.107,31.602,58.750,90.405,74.578,40.735,81.658,56.070
bsmote,70.547,45.090,39.564,55.000,94.184,74.592,46.747,80.932,57.819
ros,69.336,40.994,34.006,54.028,92.371,73.199,35.128,79.343,55.165
adasyn,74.037,40.559,31.242,62.361,89.751,76.056,40.000,83.454,57.298
casb,70.276,45.691,40.169,54.028,94.476,74.252,43.190,79.314,57.983
casb_relaxed,74.118,43.761,34.965,60.972,92.072,76.522,42.264,81.858,58.940
casb_edited,76.458,41.416,30.729,65.972,89.748,77.860,37.647,84.881,58.937
casb_ros,66.818,41.769,36.745,49.583,94.186,71.885,37.657,79.610,54.293


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,46.567,36.162,100.000,25.000,100.000,62.500,63.186,90.668,41.364
smote,64.255,46.883,48.596,45.694,96.658,71.176,54.421,90.750,55.569
bsmote,60.162,45.149,55.187,39.722,97.893,68.808,56.377,90.877,52.655
ros,63.528,52.302,67.278,44.444,98.839,71.642,58.828,90.720,57.915
adasyn,68.140,51.012,51.401,51.389,96.948,74.168,56.907,91.048,59.576
casb,63.829,49.786,58.556,44.583,98.185,71.384,54.824,90.843,56.808
casb_relaxed,65.007,50.325,56.115,46.667,97.894,72.280,59.072,91.623,57.666
casb_edited,67.004,45.427,42.650,50.417,95.346,72.881,53.106,90.988,56.215
casb_ros,65.870,54.609,66.389,47.778,98.694,73.236,60.396,91.580,60.239


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,65.314,58.152,88.333,43.611,99.710,71.661,71.347,95.540,61.733
smote,86.091,53.174,39.820,80.833,92.294,86.564,73.947,96.107,69.632
bsmote,81.981,55.773,46.714,72.361,94.624,83.493,72.449,94.515,68.877
ros,85.488,52.989,39.938,79.583,92.438,86.011,74.283,96.355,69.238
adasyn,87.137,55.608,42.173,82.083,92.948,87.516,74.049,96.079,71.373
casb,85.554,58.211,46.722,78.333,94.329,86.331,74.230,96.223,71.882
casb_relaxed,85.350,56.070,43.888,78.333,93.677,86.005,74.483,96.247,70.710
casb_edited,85.949,52.328,38.941,80.833,92.003,86.418,73.338,95.885,69.139
casb_ros,86.420,59.843,48.354,79.583,94.620,87.102,73.876,96.116,73.132


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,40.672,33.895,72.083,25.000,99.928,62.464,60.975,89.042,37.284
smote,74.337,45.900,38.448,61.875,92.967,77.421,56.241,90.161,60.118
bsmote,71.286,48.090,45.328,55.868,95.349,75.609,57.845,88.861,59.688
ros,75.203,49.209,45.097,63.021,93.986,78.503,57.176,89.933,62.206
adasyn,76.879,48.229,40.010,65.729,92.913,79.321,56.477,90.734,62.554
casb,74.996,51.664,47.291,61.528,95.293,78.410,58.618,89.985,63.330
casb_relaxed,75.868,50.249,44.256,63.507,94.402,78.954,59.178,90.660,63.058
casb_edited,77.422,46.979,37.672,66.944,92.383,79.664,55.283,91.078,62.201
casb_ros,74.181,51.866,48.700,60.590,95.493,78.042,59.263,90.284,63.024


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb,80.324,ros,82.461,-2.136,LOSS
1,f1,casb,52.967,ros,50.552,2.415,WIN
2,precision,casb,43.716,bsmote,39.846,3.871,WIN
3,recall,casb_edited,70.556,ros,74.028,-3.472,LOSS


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,76.458,adasyn,74.037,2.421,WIN
1,f1,casb,45.691,bsmote,45.090,0.601,WIN
2,precision,casb,40.169,bsmote,39.564,0.605,WIN
3,recall,casb_edited,65.972,adasyn,62.361,3.611,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,67.004,adasyn,68.140,-1.136,LOSS
1,f1,casb_ros,54.609,ros,52.302,2.307,WIN
2,precision,casb_ros,66.389,ros,67.278,-0.889,LOSS
3,recall,casb_edited,50.417,adasyn,51.389,-0.972,LOSS


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,77.422,adasyn,76.879,0.544,WIN
1,f1,casb_ros,51.866,ros,49.209,2.657,WIN
2,precision,casb_ros,48.700,bsmote,45.328,3.373,WIN
3,recall,casb_edited,66.944,adasyn,65.729,1.215,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.577350,0.500000,1.000000,0.333333,1.000000,0.666667,0.960317,0.995974,0.538675,None
1,42,1,svm,smote,0.50,0.792477,0.521739,0.428571,0.666667,0.942029,0.804348,0.711555,0.936393,0.657108,None
2,42,1,svm,bsmote,0.50,0.795519,0.545455,0.461538,0.666667,0.949275,0.807971,0.752988,0.933172,0.670487,None
3,42,1,svm,ros,0.50,0.859259,0.608696,0.500000,0.777778,0.949275,0.863527,0.842958,0.966989,0.733977,None
4,42,1,svm,adasyn,0.50,0.795519,0.545455,0.461538,0.666667,0.949275,0.807971,0.648838,0.951691,0.670487,None
5,42,1,svm,casb,0.75,0.859259,0.608696,0.500000,0.777778,0.949275,0.863527,0.771871,0.964573,0.733977,None
6,42,1,svm,casb_relaxed,0.50,0.859259,0.608696,0.500000,0.777778,0.949275,0.863527,0.772798,0.960548,0.733977,None
7,42,1,svm,casb_edited,0.50,0.852674,0.560000,0.437500,0.777778,0.934783,0.856280,0.750509,0.955717,0.706337,None
8,42,1,svm,casb_ros,0.75,0.859259,0.608696,0.500000,0.777778,0.949275,0.863527,0.770504,0.963768,0.733977,None
9,42,1,knn,original,0.50,0.942809,0.941176,1.000000,0.888889,1.000000,0.944444,0.895692,0.942432,0.941993,None
